# 03 — Model 2 (Mask R-CNN: Two-Stage CNN) & Model 3 (Mask2Former: Transformer)
**Runtime:** GPU T4 (Colab Free).

Notebook ini melatih dan mengevaluasi 2 model tambahan dari 2 keluarga arsitektur berbeda menggunakan dataset format **COCO Segmentation** (`main_coco` dan `custom_hp_coco`):
1. **Model 2 — Mask R-CNN (`maskrcnn_resnet50_fpn_v2`)** dari Torchvision (*Two-Stage CNN*, menggunakan RoIAlign grid 28×28).
2. **Model 3 — Mask2Former (`facebook/mask2former-swin-tiny-coco-instance`)** dari HuggingFace Transformers (*Transformer-based*).

Semua metrik dievaluasi dengan **pycocotools** standar (Mask mAP@50, Mask mAP@50-95, Box mAP@50, Box mAP@50-95) pada **Test Set Asli** dan **Test Set HP Anda**, serta menyimpan `results.csv` dan catatan waktu GPU ke Google Drive.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pycocotools transformers accelerate albumentations timm pandas

import os, time, json, math, glob
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from pycocotools import mask as mask_utils

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device, torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

PROJECT_DIR = '/content/drive/MyDrive/CPMK3_Instance_Segmentation'
MAIN_COCO   = os.path.join(PROJECT_DIR, 'datasets', 'main_coco')
CUSTOM_COCO = os.path.join(PROJECT_DIR, 'datasets', 'custom_hp_coco')
LOGS_DIR    = os.path.join(PROJECT_DIR, 'logs')
RUNS_DIR    = os.path.join(PROJECT_DIR, 'runs')
os.makedirs(LOGS_DIR, exist_ok=True)

def find_coco_split(root_dir, split_candidates):
    for sp in split_candidates:
        sp_dir = os.path.join(root_dir, sp)
        ann = os.path.join(sp_dir, '_annotations.coco.json')
        if os.path.exists(ann):
            return sp_dir, ann
    raise FileNotFoundError(f'Tidak menemukan _annotations.coco.json di {root_dir} ({split_candidates})')

TRAIN_DIR, TRAIN_ANN = find_coco_split(MAIN_COCO, ['train'])
VAL_DIR,   VAL_ANN   = find_coco_split(MAIN_COCO, ['valid', 'val'])
TEST_DIR,  TEST_ANN  = find_coco_split(MAIN_COCO, ['test', 'valid'])
HP_DIR,    HP_ANN    = find_coco_split(CUSTOM_COCO, ['test', 'valid', 'train'])

print('Train :', TRAIN_ANN)
print('Val   :', VAL_ANN)
print('Test  :', TEST_ANN)
print('HP    :', HP_ANN)


## Helper: Dataset COCO & Evaluasi `pycocotools`
Memetakan kategori Roboflow secara konsisten (menghindari kategori super-parent id=0 bawaan Roboflow jika ada).


In [ ]:
def get_valid_categories(ann_file):
    coco = COCO(ann_file)
    cats = coco.loadCats(coco.getCatIds())
    # Roboflow kadang menyertakan parent category dengan supercategory == 'none' dan tidak dipakai di anotasi
    used_cat_ids = set(ann['category_id'] for ann in coco.dataset['annotations'])
    valid_cats = [c for c in cats if c['id'] in used_cat_ids]
    valid_cats = sorted(valid_cats, key=lambda x: x['id'])
    cat_id_to_contig = {c['id']: i + 1 for i, c in enumerate(valid_cats)}  # 1..K (0 = background)
    contig_to_cat_id = {i + 1: c['id'] for i, c in enumerate(valid_cats)}
    id_to_name = {i + 1: c['name'] for i, c in enumerate(valid_cats)}
    return valid_cats, cat_id_to_contig, contig_to_cat_id, id_to_name

VALID_CATS, CAT2CONTIG, CONTIG2CAT, ID2NAME = get_valid_categories(TRAIN_ANN)
NUM_CLASSES = len(VALID_CATS) + 1  # +1 background untuk Mask R-CNN
print('Kelas terdeteksi:', ID2NAME, '| NUM_CLASSES (inc. bg):', NUM_CLASSES)


class CocoInstanceDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig, img_size=640):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.img_size = img_size
        # Pastikan hanya gambar yang punya minimal 1 anotasi valid saat training
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img_path = os.path.join(self.img_dir, info['file_name'])
        img = Image.open(img_path).convert('RGB')
        orig_w, orig_h = img.size

        img_resized = img.resize((self.img_size, self.img_size), Image.BILINEAR)
        img_tensor = torchvision.transforms.functional.to_tensor(img_resized)

        ann_ids = self.coco.getAnnIds(imgIds=img_id)
        anns = self.coco.loadAnns(ann_ids)

        boxes, labels, masks = [], [], []
        sx, sy = self.img_size / orig_w, self.img_size / orig_h

        for ann in anns:
            if ann['category_id'] not in self.cat2contig:
                continue
            x, y, w, h = ann['bbox']
            if w <= 1 or h <= 1:
                continue
            m = self.coco.annToMask(ann)
            m_resized = np.array(
                Image.fromarray(m).resize((self.img_size, self.img_size), Image.NEAREST),
                dtype=np.uint8,
            )
            if m_resized.sum() == 0:
                continue
            boxes.append([x * sx, y * sy, (x + w) * sx, (y + h) * sy])
            labels.append(self.cat2contig[ann['category_id']])
            masks.append(m_resized)

        if len(boxes) == 0:
            boxes_t = torch.zeros((0, 4), dtype=torch.float32)
            labels_t = torch.zeros((0,), dtype=torch.int64)
            masks_t = torch.zeros((0, self.img_size, self.img_size), dtype=torch.uint8)
        else:
            boxes_t = torch.tensor(boxes, dtype=torch.float32)
            labels_t = torch.tensor(labels, dtype=torch.int64)
            masks_t = torch.tensor(np.stack(masks, axis=0), dtype=torch.uint8)

        target = {
            'boxes': boxes_t,
            'labels': labels_t,
            'masks': masks_t,
            'image_id': torch.tensor([img_id]),
            'orig_size': (orig_h, orig_w),
        }
        return img_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def append_gpu_log(row_dict):
    gpu_log_path = os.path.join(LOGS_DIR, 'gpu_time_log.csv')
    df_new = pd.DataFrame([row_dict])
    if os.path.exists(gpu_log_path):
        df_all = pd.concat([pd.read_csv(gpu_log_path), df_new], ignore_index=True)
    else:
        df_all = df_new
    df_all.to_csv(gpu_log_path, index=False)


## BAGIAN 1: Training & Evaluasi Model 2 — Mask R-CNN (Two-Stage CNN)
Untuk eksperimen NRP 0, Anda bisa menjalankan `MASKRCNN_SIZES = [640]` terlebih dahulu (baseline), lalu tambah `[320, 960]` jika kuota GPU masih cukup.


In [ ]:
def build_maskrcnn(num_classes, img_size=640):
    model = maskrcnn_resnet50_fpn_v2(
        weights=MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT,
        min_size=img_size,
        max_size=img_size,
    )
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    return model


@torch.no_grad()
def evaluate_maskrcnn_coco(model, img_dir, ann_file, img_size=640, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    # Petakan nama kelas di ann_file ke category_id ann_file (antisipasi perbedaan ID antara main & custom)
    name_to_ann_catid = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}

    results_bbox, results_segm = [], []
    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        img_resized = img.resize((img_size, img_size), Image.BILINEAR)
        tensor = torchvision.transforms.functional.to_tensor(img_resized).to(device)

        out = model([tensor])[0]
        scores = out['scores'].cpu().numpy()
        labels = out['labels'].cpu().numpy()
        boxes  = out['boxes'].cpu().numpy()
        masks  = out['masks'].cpu().numpy()  # (N, 1, H, W)

        sx, sy = orig_w / img_size, orig_h / img_size
        for s, l, b, m in zip(scores, labels, boxes, masks):
            if s < score_thresh:
                continue
            cls_name = ID2NAME.get(int(l))
            if cls_name not in name_to_ann_catid:
                continue
            cat_id = name_to_ann_catid[cls_name]
            x1, y1, x2, y2 = b[0] * sx, b[1] * sy, b[2] * sx, b[3] * sy
            results_bbox.append({
                'image_id': int(img_id),
                'category_id': int(cat_id),
                'bbox': [float(x1), float(y1), float(x2 - x1), float(y2 - y1)],
                'score': float(s),
            })
            bin_mask = (m[0] >= 0.5).astype(np.uint8)
            bin_mask_orig = np.array(
                Image.fromarray(bin_mask).resize((orig_w, orig_h), Image.NEAREST),
                dtype=np.uint8,
            )
            rle = mask_utils.encode(np.asfortranarray(bin_mask_orig))
            rle['counts'] = rle['counts'].decode('utf-8')
            results_segm.append({
                'image_id': int(img_id),
                'category_id': int(cat_id),
                'segmentation': rle,
                'score': float(s),
            })

    metrics = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if len(results_bbox) > 0:
        coco_dt = coco_gt.loadRes(results_bbox)
        ev = COCOeval(coco_gt, coco_dt, 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['box_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['box_mAP50']    = round(float(ev.stats[1]), 4)
    if len(results_segm) > 0:
        coco_dt = coco_gt.loadRes(results_segm)
        ev = COCOeval(coco_gt, coco_dt, 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['mask_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['mask_mAP50']    = round(float(ev.stats[1]), 4)
    return metrics


# === KONFIGURASI TRAINING MASK R-CNN ===
MASKRCNN_EPOCHS = 15
MASKRCNN_SIZES  = [640]  # Ubah ke [320, 640, 960] jika ingin uji ketiga resolusi di Mask R-CNN
MASKRCNN_BATCH  = {320: 4, 640: 4, 960: 2}

mr_eval_rows = []

for imgsz in MASKRCNN_SIZES:
    run_dir = os.path.join(RUNS_DIR, 'maskrcnn', f'maskrcnn_img{imgsz}')
    os.makedirs(run_dir, exist_ok=True)
    ckpt_path = os.path.join(run_dir, 'best.pth')
    csv_path  = os.path.join(run_dir, 'results.csv')

    train_ds = CocoInstanceDataset(TRAIN_DIR, TRAIN_ANN, CAT2CONTIG, img_size=imgsz)
    train_loader = DataLoader(train_ds, batch_size=MASKRCNN_BATCH[imgsz], shuffle=True,
                              num_workers=2, collate_fn=collate_fn)

    model = build_maskrcnn(NUM_CLASSES, img_size=imgsz).to(device)
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=1e-4, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    if not os.path.exists(ckpt_path):
        t0 = time.time()
        history = []
        best_loss = float('inf')

        for epoch in range(1, MASKRCNN_EPOCHS + 1):
            model.train()
            ep_loss = 0.0
            n_steps = 0
            ep_t0 = time.time()
            for imgs, targets in train_loader:
                # Lewati batch yang tidak memiliki objek
                valid_pairs = [(im.to(device), {k: v.to(device) if torch.is_tensor(v) else v for k, v in tg.items()})
                               for im, tg in zip(imgs, targets) if len(tg['boxes']) > 0]
                if not valid_pairs:
                    continue
                im_list, tg_list = zip(*valid_pairs)

                optimizer.zero_grad()
                with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                    loss_dict = model(list(im_list), list(tg_list))
                    losses = sum(loss for loss in loss_dict.values())

                scaler.scale(losses).backward()
                scaler.step(optimizer)
                scaler.update()

                ep_loss += float(losses.item())
                n_steps += 1

            avg_loss = ep_loss / max(n_steps, 1)
            ep_sec = round(time.time() - ep_t0, 1)
            print(f'[Mask R-CNN img={imgsz}] Epoch {epoch}/{MASKRCNN_EPOCHS} | loss={avg_loss:.4f} | {ep_sec}s')
            history.append({'epoch': epoch, 'train_loss': round(avg_loss, 4), 'epoch_sec': ep_sec})

            if avg_loss < best_loss:
                best_loss = avg_loss
                torch.save(model.state_dict(), ckpt_path)

        pd.DataFrame(history).to_csv(csv_path, index=False)
        elapsed_min = round((time.time() - t0) / 60.0, 2)
        append_gpu_log({
            'model': 'MaskRCNN-R50-FPNv2',
            'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz,
            'epochs': MASKRCNN_EPOCHS,
            'batch_size': MASKRCNN_BATCH[imgsz],
            'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
            'train_time_minutes': elapsed_min,
            'results_csv': csv_path,
        })
    else:
        print(f'[SKIP TRAIN] Memuat checkpoint tersimpan: {ckpt_path}')

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
        t0 = time.time()
        m = evaluate_maskrcnn_coco(model, d_dir, d_ann, img_size=imgsz)
        m.update({
            'model': 'MaskRCNN-R50-FPNv2',
            'architecture_family': 'Two-Stage CNN',
            'imgsz': imgsz,
            'dataset': split_label,
            'eval_time_sec': round(time.time() - t0, 2),
        })
        mr_eval_rows.append(m)

    del model
    torch.cuda.empty_cache()

df_mr = pd.DataFrame(mr_eval_rows)
df_mr.to_csv(os.path.join(LOGS_DIR, 'eval_maskrcnn.csv'), index=False)
df_mr


## BAGIAN 2: Training & Evaluasi Model 3 — Mask2Former (Transformer)
Menggunakan `facebook/mask2former-swin-tiny-coco-instance` dengan `fp16` dan `batch_size=2` (+ gradient accumulation 4 langkah -> efektif batch 8) agar hemat VRAM di GPU T4 Colab Free.


In [ ]:
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation

M2F_MODEL_ID = 'facebook/mask2former-swin-tiny-coco-instance'
M2F_EPOCHS   = 15
M2F_IMG_SIZE = 640   # Baseline 640 (bisa diubah ke 320 / 960 jika ingin uji NRP 0 di Mask2Former juga)
M2F_BATCH    = 2
GRAD_ACCUM   = 4

# Label untuk Mask2Former: 0..K-1 (tanpa background di id2label)
M2F_ID2LABEL = {i: c['name'] for i, c in enumerate(VALID_CATS)}
M2F_LABEL2ID = {c['name']: i for i, c in enumerate(VALID_CATS)}
M2F_CAT2CONTIG = {c['id']: i for i, c in enumerate(VALID_CATS)}

processor = AutoImageProcessor.from_pretrained(
    M2F_MODEL_ID,
    do_reduce_labels=False,
    size={'height': M2F_IMG_SIZE, 'width': M2F_IMG_SIZE},
)


class Mask2FormerCocoDataset(Dataset):
    def __init__(self, img_dir, ann_file, cat2contig):
        self.img_dir = img_dir
        self.coco = COCO(ann_file)
        self.cat2contig = cat2contig
        self.img_ids = sorted(self.coco.getImgIds())

    def __len__(self):
        return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs(img_id)[0]
        img = Image.open(os.path.join(self.img_dir, info['file_name'])).convert('RGB')
        w, h = img.size

        inst_map = np.zeros((h, w), dtype=np.int32)
        inst2class = {}
        inst_id = 1
        for ann in self.coco.loadAnns(self.coco.getAnnIds(imgIds=img_id)):
            if ann['category_id'] not in self.cat2contig:
                continue
            m = self.coco.annToMask(ann)
            if m.sum() == 0:
                continue
            inst_map[m == 1] = inst_id
            inst2class[inst_id] = self.cat2contig[ann['category_id']]
            inst_id += 1

        if len(inst2class) == 0:
            # Dummy kosong (akan di-skip saat training)
            return None

        inputs = processor(
            images=[img],
            segmentation_maps=[inst_map],
            instance_id_to_semantic_id=inst2class,
            return_tensors='pt',
        )
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'pixel_mask': inputs['pixel_mask'].squeeze(0),
            'mask_labels': inputs['mask_labels'][0],
            'class_labels': inputs['class_labels'][0],
        }


def m2f_collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if not batch:
        return None
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'pixel_mask': torch.stack([b['pixel_mask'] for b in batch]),
        'mask_labels': [b['mask_labels'] for b in batch],
        'class_labels': [b['class_labels'] for b in batch],
    }


m2f_run_dir = os.path.join(RUNS_DIR, 'mask2former', f'm2f_swin_tiny_img{M2F_IMG_SIZE}')
os.makedirs(m2f_run_dir, exist_ok=True)
m2f_ckpt_dir = os.path.join(m2f_run_dir, 'best_model')
m2f_csv_path = os.path.join(m2f_run_dir, 'results.csv')

if not os.path.exists(m2f_ckpt_dir):
    m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(
        M2F_MODEL_ID,
        id2label=M2F_ID2LABEL,
        label2id=M2F_LABEL2ID,
        ignore_mismatched_sizes=True,
    ).to(device)

    train_ds = Mask2FormerCocoDataset(TRAIN_DIR, TRAIN_ANN, M2F_CAT2CONTIG)
    train_loader = DataLoader(train_ds, batch_size=M2F_BATCH, shuffle=True,
                              num_workers=2, collate_fn=m2f_collate_fn)

    optimizer = torch.optim.AdamW(m2f_model.parameters(), lr=5e-5, weight_decay=1e-4)
    scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    t0 = time.time()
    history = []
    best_loss = float('inf')

    for epoch in range(1, M2F_EPOCHS + 1):
        m2f_model.train()
        ep_loss, n_steps = 0.0, 0
        ep_t0 = time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader, 1):
            if batch is None:
                continue
            pixel_values = batch['pixel_values'].to(device)
            pixel_mask   = batch['pixel_mask'].to(device)
            mask_labels  = [m.to(device) for m in batch['mask_labels']]
            class_labels = [c.to(device) for c in batch['class_labels']]

            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                outputs = m2f_model(
                    pixel_values=pixel_values,
                    pixel_mask=pixel_mask,
                    mask_labels=mask_labels,
                    class_labels=class_labels,
                )
                loss = outputs.loss / GRAD_ACCUM

            scaler.scale(loss).backward()
            if step % GRAD_ACCUM == 0 or step == len(train_loader):
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()

            ep_loss += float(outputs.loss.item())
            n_steps += 1

        avg_loss = ep_loss / max(n_steps, 1)
        ep_sec = round(time.time() - ep_t0, 1)
        print(f'[Mask2Former img={M2F_IMG_SIZE}] Epoch {epoch}/{M2F_EPOCHS} | loss={avg_loss:.4f} | {ep_sec}s')
        history.append({'epoch': epoch, 'train_loss': round(avg_loss, 4), 'epoch_sec': ep_sec})

        if avg_loss < best_loss:
            best_loss = avg_loss
            m2f_model.save_pretrained(m2f_ckpt_dir)

    pd.DataFrame(history).to_csv(m2f_csv_path, index=False)
    elapsed_min = round((time.time() - t0) / 60.0, 2)
    append_gpu_log({
        'model': 'Mask2Former-Swin-Tiny',
        'architecture_family': 'Transformer',
        'imgsz': M2F_IMG_SIZE,
        'epochs': M2F_EPOCHS,
        'batch_size': M2F_BATCH * GRAD_ACCUM,
        'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
        'train_time_minutes': elapsed_min,
        'results_csv': m2f_csv_path,
    })
else:
    print(f'[SKIP TRAIN] Memuat Mask2Former tersimpan: {m2f_ckpt_dir}')
    m2f_model = Mask2FormerForUniversalSegmentation.from_pretrained(m2f_ckpt_dir).to(device)


@torch.no_grad()
def evaluate_mask2former_coco(model, img_dir, ann_file, score_thresh=0.05):
    model.eval()
    coco_gt = COCO(ann_file)
    name_to_ann_catid = {c['name']: c['id'] for c in coco_gt.loadCats(coco_gt.getCatIds())}
    results_bbox, results_segm = [], []

    for img_id in coco_gt.getImgIds():
        info = coco_gt.loadImgs(img_id)[0]
        img = Image.open(os.path.join(img_dir, info['file_name'])).convert('RGB')
        orig_w, orig_h = img.size
        inputs = processor(images=img, return_tensors='pt').to(device)
        outputs = model(**inputs)
        pred = processor.post_process_instance_segmentation(
            outputs, threshold=score_thresh, target_sizes=[(orig_h, orig_w)]
        )[0]
        seg_map = pred['segmentation'].cpu().numpy()
        for seg_info in pred['segments_info']:
            cls_name = M2F_ID2LABEL.get(int(seg_info['label_id']))
            if cls_name not in name_to_ann_catid:
                continue
            cat_id = name_to_ann_catid[cls_name]
            bin_mask = (seg_map == seg_info['id']).astype(np.uint8)
            if bin_mask.sum() == 0:
                continue
            ys, xs = np.where(bin_mask == 1)
            x1, y1, x2, y2 = float(xs.min()), float(ys.min()), float(xs.max()), float(ys.max())
            score = float(seg_info['score'])
            results_bbox.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': score,
            })
            rle = mask_utils.encode(np.asfortranarray(bin_mask))
            rle['counts'] = rle['counts'].decode('utf-8')
            results_segm.append({
                'image_id': int(img_id), 'category_id': int(cat_id),
                'segmentation': rle, 'score': score,
            })

    metrics = {'box_mAP50': 0.0, 'box_mAP50_95': 0.0, 'mask_mAP50': 0.0, 'mask_mAP50_95': 0.0}
    if len(results_bbox) > 0:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_bbox), 'bbox')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['box_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['box_mAP50']    = round(float(ev.stats[1]), 4)
    if len(results_segm) > 0:
        ev = COCOeval(coco_gt, coco_gt.loadRes(results_segm), 'segm')
        ev.evaluate(); ev.accumulate(); ev.summarize()
        metrics['mask_mAP50_95'] = round(float(ev.stats[0]), 4)
        metrics['mask_mAP50']    = round(float(ev.stats[1]), 4)
    return metrics


m2f_eval_rows = []
for split_label, d_dir, d_ann in [('Test_Asli', TEST_DIR, TEST_ANN), ('Test_HP_Sendiri', HP_DIR, HP_ANN)]:
    t0 = time.time()
    m = evaluate_mask2former_coco(m2f_model, d_dir, d_ann)
    m.update({
        'model': 'Mask2Former-Swin-Tiny',
        'architecture_family': 'Transformer',
        'imgsz': M2F_IMG_SIZE,
        'dataset': split_label,
        'eval_time_sec': round(time.time() - t0, 2),
    })
    m2f_eval_rows.append(m)

df_m2f = pd.DataFrame(m2f_eval_rows)
df_m2f.to_csv(os.path.join(LOGS_DIR, 'eval_mask2former.csv'), index=False)
df_m2f
